In [ ]:
#| default_exp download

# Download helpers

- Download and process LLM-ready documents

In [ ]:
#| export
from fastcore.utils import *
from httpx import get
from fastcore.meta import delegates
from urllib.parse import urlparse, urljoin
from textwrap import dedent

In [ ]:
from IPython.display import Markdown,HTML
from fastcore.test import *

In [ ]:
#| export
def clean_md(text, rm_comments=True, rm_details=True):
    "Remove comments and `<details>` sections from `text`"
    if rm_comments: text = re.sub(r'\n?<!--.*?-->\n?', '', text, flags=re.DOTALL)
    if rm_details: text = re.sub(r'\n?<details>.*?</details>\n?', '', text, flags=re.DOTALL)
    return text

In [ ]:
#| export
@delegates(get)
def read_md(url, rm_comments=True, rm_details=True, **kwargs):
    "Read text from `url` and clean with `clean_docs`"
    return clean_md(get(url, **kwargs).text, rm_comments=rm_comments, rm_details=rm_details)

In [ ]:
mdurl = 'https://claudette.answer.ai/index.html.md'
md = read_md(mdurl)
# Markdown(md)

In [ ]:
#| export
def html2md(
    s:str, # HTML to convert
    ignore_links=True, # Drop links, keeping their text
    ignore_images=True, # Drop images
    base_url:str='' # URL that relative link and image URLs resolve against
):
    "Convert `s` from HTML to markdown, with code blocks fenced"
    import html2text
    o = html2text.HTML2Text(baseurl=base_url, bodywidth=0)
    o.ignore_links,o.ignore_images,o.mark_code = ignore_links,ignore_images,True
    md = o.handle(s)
    return re.sub(r'\[code]\s*\n(.*?)\n\[/code]', lambda m: f'````\n{dedent(m.group(1))}\n````', md, flags=re.DOTALL).strip()

`html2md` fences code blocks, and resolves relative link and image URLs against `base_url` when it keeps them:

In [ ]:
s = '<p>See <a href="b.html">this</a> <img alt="fig" src="f.png"></p><pre><code>  x = 1\n  y = 2</code></pre>'
md = html2md(s, ignore_links=False, ignore_images=False, base_url='https://ex.org/a/')
assert '[this](https://ex.org/a/b.html)' in md and '![fig](https://ex.org/a/f.png)' in md
assert '````\nx = 1\ny = 2\n````' in md
md

In [ ]:
#| export
def read_html(
    url, # URL to read
    sel=None, # Read only outerHTML of CSS selector `sel`
    rm_comments=True, # Removes HTML comments
    rm_details=True, # Removes `<details>` tags
    multi=False, # Get all matches to `sel` or first one
    wrap_tag=None, #If multi, each selection wrapped with <wrap_tag>content</wrap_tag>
    ignore_links=True,
): # Cleaned markdown
    "Get `url`, optionally selecting CSS selector `sel`, and convert to clean markdown"
    page = get(url).text
    if sel:
        from bs4 import BeautifulSoup
        soup = BeautifulSoup(page, 'html.parser')
        if multi:
            page = [str(el) for el in soup.select(sel)]
            if not wrap_tag: page = "\n".join(page)
        else: page = str(soup.select_one(sel))
    mds = map(lambda x: clean_md(html2md(x, ignore_links=ignore_links), rm_comments, rm_details=rm_details), tuplify(page))
    if wrap_tag: return '\n'.join([f"\n<{wrap_tag}>\n{o}</{wrap_tag}>\n" for o in mds])
    else: return'\n'.join(mds)

In [ ]:
# test single class selector
listings = read_html('https://www.answer.ai/', sel='.listing-description')
assert len(listings) < 500

# Test multi class selector
listings = read_html('https://www.answer.ai/', sel='.listing-description', multi=True)
assert len(listings) > 1000 # returns more than single so selecting multi

# Test multi_wrap_tag
listings = read_html('https://www.answer.ai/', sel='.listing-description', multi=True, wrap_tag='document')
assert '<document>' in listings and '</document>' in listings 

In [ ]:
read_html('https://www.answer.ai/', sel='.listing-description', ignore_links=False)

'[ How data quality, proxy measures, and missing domain expertise can undermine AI science results ](./posts/2026-03-17-risks-life-sciences/index.html)\n\n'

In [ ]:
# test tag css selectors
assert len(read_html('https://www.answer.ai/', sel='div.listing-description', multi=True)) > 1000
assert len(read_html('https://www.answer.ai/', sel='div', multi=True)) > 1000

In [ ]:
htmlurl = 'https://hypermedia.systems/hypermedia-a-reintroduction/'
hmd = read_html(htmlurl)
assert len(hmd) > 100
# Markdown(hmd)

`read_url` reads a page the way a browser would, through `scrape_url`, which gets past Cloudflare's bot checks. It converts HTML to markdown with `html2md`, keeping links and images with absolute URLs. A URL fragment, such as `#unit-files`, selects just that section, and `selector` selects elements by CSS selector instead. Content that isn't HTML, such as an `llms.txt` file, comes back unchanged.

In [ ]:
#| export
def scrape_url(url):
    "Get `url` with a Cloudflare-aware scraper, decoding text without a declared encoding as UTF-8"
    from cloudscraper import create_scraper
    o = create_scraper().get(url)
    if not o.encoding or o.encoding == 'ISO-8859-1': o.encoding = 'utf-8'
    return o

In [ ]:
#| export
def _extract_section(soup, url, selector=None):
    "HTML of the elements matching `selector`, or of the section starting at `url`'s fragment, or the whole page"
    if selector: return '\n\n'.join(str(s) for s in soup.select(selector))
    frag = urlparse(url).fragment
    if not frag: return str(soup)
    section = soup.find(id=frag)
    if not section: return ''
    elements,current = [section],section.next_sibling
    while current:
        if getattr(current, 'name', None) == section.name: break
        elements.append(current)
        current = current.next_sibling
    return ''.join(str(el) for el in elements)

html2text escapes every backslash that precedes punctuation. In TeX, that turns `\{` into `\\{`. To keep each formula intact, `_tex_math` replaces it with a placeholder before conversion. `read_url` puts the TeX back afterwards.

In [ ]:
#| export
def _tex_math(soup):
    "Replace each MathML element in `soup` that carries TeX with a placeholder, returning that TeX as inline or display Markdown math"
    texs = []
    for m in soup.find_all('math'):
        if not (a := m.find('annotation', encoding='application/x-tex')): continue
        tex = a.text.strip()
        texs.append(f'$${tex}$$' if m.get('display') == 'block' else rf'\({tex}\)')
        m.replace_with(f'MATHTEX{len(texs)-1}X')
    return texs

In [ ]:
#| export
def read_url(
    url:str, # URL to read
    as_md:bool=True, # Convert HTML to markdown
    extract_section:bool=True, # Extract section matching URL fragment or selector
    selector:str=None, # CSS selector to extract specific content
):
    "Read `url`, converting HTML to markdown with absolute link and image URLs"
    from bs4 import BeautifulSoup
    o = scrape_url(url)
    if (o.headers.get('content-type') or 'text/plain').split(';')[0] != 'text/html': return o.text
    soup = BeautifulSoup(o.text, 'lxml')
    if ('#' in url and extract_section) or selector: soup = BeautifulSoup(_extract_section(soup, url, selector), 'lxml')
    if not as_md: return str(soup)
    texs = _tex_math(soup)
    base = soup.find('base')
    md = html2md(str(soup), ignore_links=False, ignore_images=False, base_url=urljoin(url, base['href'] if base else ''))
    return re.sub(r'MATHTEX(\d+)X', lambda m: texs[int(m[1])], md)

In [ ]:
read_url('https://www.example.org')

A fragment selects the section that starts at that id, up to the next element of the same kind:

In [ ]:
sec = read_url('https://caddyserver.com/docs/running#unit-files')
assert sec.startswith('### Unit Files')
sec[:300]

Formulas in MathML that carry their TeX source, as in arXiv's HTML papers, come back as TeX, written `\(...\)` inline and `$$...$$` for display math:

In [ ]:
att = read_url('https://arxiv.org/html/1706.03762v7#S3.SS2.SSS1')
assert r'$$\mathrm{Attention}(Q,K,V)' in att and r'\(d_{k}\)' in att
Markdown(att)

Plain text, such as an `llms.txt` file, keeps its own line breaks:

In [ ]:
read_url('https://fastht.ml/docs/llms.txt').splitlines()[:3]

In [ ]:
#| export
def get_llmstxt(url, optional=False, n_workers=None):
    "Get llms.txt file from and expand it with `llms_txt.create_ctx()`"
    if not url.endswith('llms.txt'): return None
    import llms_txt
    resp = get(url)
    if resp.status_code!=200: return None
    return llms_txt.create_ctx(resp.text, optional=optional, n_workers=n_workers)

In [ ]:
# print(get_llmstxt('https://llmstxt.org/llms.txt'))

In [ ]:
#| export
def split_url(url):
    "Split `url` into base, path, and file name, normalising name to '/' if empty"
    parsed = urlparse(url.strip('/'))
    base = f"{parsed.scheme}://{parsed.netloc}"
    path,spl,fname = parsed.path.rpartition('/')
    fname = spl+fname
    if not path and not fname: path='/'
    return base,path,fname

In [ ]:
urls = ('https://claudette.answer.ai/path/', 'https://claudette.answer.ai/', 'https://llmstxt.org', 'https://llmstxt.org/')

[split_url(o) for o in urls]

[('https://claudette.answer.ai', '', '/path'),
 ('https://claudette.answer.ai', '/', ''),
 ('https://llmstxt.org', '/', ''),
 ('https://llmstxt.org', '/', '')]

In [ ]:
#| export
def _tryget(url):
    "Return response from `url` if `status_code!=404`, otherwise `None`"
    res = get(url)
    return None if res.status_code==404 else url

In [ ]:
#| export
def find_docs(url):
    "If available, return LLM-friendly llms.txt context or markdown file location from `url`"
    base,path,fname = split_url(url)
    url = (base+path+fname).strip('/')
    if fname=='/llms.txt': return url
    if Path(fname).suffix in('.md', '.txt', '.rst'): return _tryget(url)
    if '.' in fname: return _tryget(url+'.md') or find_docs(url[:url.rfind('/')])
    res = _tryget(url+'/llms.txt')
    if res: return res
    res = _tryget(url+'/index.md')
    if res: return res
    res = _tryget(url+'/index.html.md')
    if res: return res
    res = _tryget(url+'/index-commonmark.md')
    if res: return res
    parsed_url = urlparse(url)
    if parsed_url.path == '/' or not parsed_url.path: return None
    return find_docs(urljoin(url, '..'))

In [ ]:
fl_url = 'https://answerdotai.github.io/fastlite'

In [ ]:
find_docs(fl_url)

'https://answerdotai.github.io/fastlite/llms.txt'

In [ ]:
#| eval: false
for o in urls: print(find_docs(o))

https://claudette.answer.ai/llms.txt
https://claudette.answer.ai/llms.txt


https://llmstxt.org/llms.txt
https://llmstxt.org/llms.txt


In [ ]:
#| eval: false
suffixes = ["/", "/tmp", "/tmp/tmp/"]
for suff in suffixes:
    for o in urls:  test_eq(find_docs(o), find_docs(o+suff))

test_eq(find_docs("https://github.com"), "https://github.com/llms.txt")
test_eq(find_docs("https://github.com/AnswerDotAI"), "https://github.com/llms.txt")
test_eq(find_docs("https://github.com/AnswerDotAI/"), "https://github.com/llms.txt")

In [ ]:
#| export
def read_docs(url, optional=False, n_workers=None, rm_comments=True, rm_details=True):
    "If available, return LLM-friendly llms.txt context or markdown file response for `url`"
    url = find_docs(url)
    if url.endswith('/llms.txt'): res = get_llmstxt(url, optional=optional, n_workers=n_workers)
    else: res = get(url).text
    return clean_md(res, rm_comments=rm_comments, rm_details=rm_details)

## Export -

In [ ]:
#|hide
#|eval: false
from nbdev.doclinks import nbdev_export
nbdev_export()